In [ ]:
# %%
# Cell 1. 全局配置 & 输出目录策略
# =========================
import os
from pathlib import Path

# —— 固定：Round1（LoRA合并→推理）所在模型目录 —— 
MODEL_DIR = ""

# —— Ground Truth —— 
LABEL_CSV_PATH = ""

# —— Trace 源（真/无 bug）——
BUG_FILE      = "../data_test/actions_br_combined_mod.json"
BUG_FREE_FILE = "../data_test/test_edited_actions_mod.json"
N_SAMPLES_EACH = 51

# —— CUDA —— 
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# —— Rounds 2-4 输入源（仅对 3/4 轮生效；Round2 固定用 reason）——
R3_INPUT_SOURCE = "reason"   # 可选："generated" | "reason"

# —— 当前运行标签（只改这一行即可切换 run1/run2/run3）——
RUN_NAME = "diff_models_temp0"   # ← 你只需在不同 notebook 改这一行

# —— 输出目录规则：把 MODEL_DIR 路径中的 `test0923` 替换为 `test0923_output`
# —— 并在末尾添加 run_name + R3_INPUT_SOURCE 区分
def derive_output_dir(model_dir: str, run_name: str, r3_input_source: str) -> Path:
    if "test0923" in model_dir:
        base_out = Path(model_dir.replace("test0923", "test0923_output"))
    else:
        base_out = Path(model_dir) / "outputs"
    return base_out / f"{run_name}_{r3_input_source}"

OUTPUT_DIR = derive_output_dir(MODEL_DIR, RUN_NAME, R3_INPUT_SOURCE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[IO] MODEL_DIR       = {MODEL_DIR}")
print(f"[IO] R3_INPUT_SOURCE = {R3_INPUT_SOURCE}")
print(f"[IO] OUTPUT_DIR       = {OUTPUT_DIR}")


[IO] MODEL_DIR       = /home/yang3j7/NCF0729/test0923/sft_qwen3
[IO] R34_INPUT_SOURCE = reason
[IO] OUTPUT_DIR       = /home/yang3j7/NCF0729/test0923_output/sft_qwen3/diff_models_temp0_reason


In [2]:
# Cell 2. Round1：LoRA 合并→推理（保持你的逻辑与 Prompt 不变）
# =========================
import os, json
from tqdm import tqdm
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from peft import PeftModel

# 推理输出文件（优雅命名）
R1_JSON = OUTPUT_DIR / "r1_infer.json"

# 生成参数
MAX_SEQ_LEN  = 16384
MAX_NEW_TOK  = 2048
TOP_P        = 0.9
temperature  = 0.0

# 1) base
base_model = "unsloth/Qwen3-14B"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)

# 2) 加载 LoRA 适配器并合并（Round1 固定）
adapter_path = MODEL_DIR
model = PeftModel.from_pretrained(model, adapter_path)
model = model.merge_and_unload()
model.eval()
FastLanguageModel.for_inference(model)

# 3) 准备样本（读取 trace）
# N_SAMPLES_EACH = 51
ds_bug   = load_dataset("json", data_files=BUG_FILE, split="train").select(range(N_SAMPLES_EACH))
ds_clean = load_dataset("json", data_files=BUG_FREE_FILE, split="train").select(range(N_SAMPLES_EACH))
samples = [{"id": f"tr_{s['id']}", "actions": s["actions_content"]} for s in ds_bug] + \
          [{"id": f"fl_{s['id']}", "actions": s["actions_content"]} for s in ds_clean]
print(f"[Round1] read {len(samples)} records")

# 4) Prompt（保持不变）
system_msg = {"role": "system", "content": "You are a helpful assistant."}
prompt_tpl = """### Instruction:
Please determine whether there is a non-crash functional bug in the following android app UI interaction trace, and briefly explain the reason.

### Input:
{actions}

### Output format:
{{
  "is_bug": "Yes" or "No",
  "reason": "brief reasons for why consider as bug or no bug."
}}
"""

# 5) 推理
results = []
for s in tqdm(samples, desc="Round1 Infer"):
    user_msg = {"role": "user", "content": prompt_tpl.format(actions=s["actions"])}
    prompt = tokenizer.apply_chat_template([system_msg, user_msg], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOK,
            temperature    = temperature,
            top_p          = TOP_P,
            do_sample      = (temperature > 0),
            eos_token_id   = tokenizer.eos_token_id,
            pad_token_id   = tokenizer.eos_token_id,
        )
    gen_ids  = out[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    results.append({"id": s["id"], "generated": response})

# 6) 保存
with open(R1_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"[Round1] Saved infer JSON → {R1_JSON}")


/home/yang3j7/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.8.5: Fast Qwen3 patching. Transformers: 4.55.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 6/6 [00:05<00:00,  1.11it/s]


[Round1] read 102 records


Round1 Infer:  41%|████      | 42/102 [02:09<02:49,  2.82s/it]Unsloth: Input IDs of length 17807 > the model's max sequence length of 16384.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Round1 Infer:  91%|█████████ | 93/102 [06:39<00:26,  2.97s/it]Unsloth: Input IDs of length 17347 > the model's max sequence length of 16384.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Round1 Infer: 100%|██████████| 102/102 [09:21<00:00,  5.51s/it]

[Round1] Saved infer JSON → /home/yang3j7/NCF0729/test0923_output/sft_qwen3/diff_models_temp0_reason/r1_infer.json


In [3]:
# 清理 Round1 模型和缓存
import gc
del model, tokenizer
torch.cuda.empty_cache()
gc.collect()
print("[Round1] 模型与缓存已释放。")


[Round1] 模型与缓存已释放。


In [ ]:
# Cell 3. Round1 Judge（GPT-4o）→ 初始标签 & labels_dict
# =========================
import re, json, pandas as pd
from tqdm import tqdm
from openai import OpenAI

R1_CSV = OUTPUT_DIR / "r1_judge.csv"

openai_api_key = ""
client = OpenAI(api_key=openai_api_key)
model_name = "gpt-4o"

def extract_is_bug_and_reason_fallback(text: str):
    bug_match = re.search(
        r'"{0,2}(?:is_bug|is_ncf_bug|BugExist[ae]nceJudgment)"{0,2}\s*:\s*["“”]?(yes|no|true|false|1|0)["“”]?',
        text, flags=re.I
    )
    is_bug = None
    if bug_match:
        val = bug_match.group(1).strip().lower()
        is_bug = val in ("yes","true","1")
    reason_match = re.search(
        r'"{0,2}(?:reason|BugReasoningExplanation|explanation)"{0,2}\s*:\s*["“”]?(.*?)["“”]?(?:,|\n|\}|$)',
        text, flags=re.I | re.S
    )
    reason = reason_match.group(1).strip() if reason_match else ""
    return is_bug, reason

judge_prompt_tpl = """You are given two pieces of information about a potential bug in an app's operation:

1. Official bug label:
- Is Bug: {is_bug}
- Reason: {reason}

2. AI-generated output:
- Output: {output}

Task: Determine if the AI output agrees with the official bug label and reason. Focus on whether the AI's reasoning matches the official reason and whether it correctly reflects the bug/non-bug status.

If they agree:
Start your answer with:
"Yes. They agree."
Then explain briefly why.

If they do NOT agree:
Start your answer with:
"No. They do not agree."
Then explain briefly why.
"""

# 评测 Round1
with open(R1_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)
gt_df = pd.read_csv(LABEL_CSV_PATH, dtype={"id": str})
gt_map = gt_df.set_index("id").to_dict("index")

rows = []
# skip_ids = {"5","16","37"}
skip_ids = {"43"}
print(skip_ids)
for item in tqdm(data, desc="Round1 Judge"):
    _id = item["id"]
    true_id = _id.split("_",1)[-1]
    if true_id in skip_ids:
        continue
    raw = item["generated"]
    ai_is_bug, ai_reason = extract_is_bug_and_reason_fallback(raw)
    if ai_is_bug is None:
        rows.append({"id":_id,"label":"parse_error","ai_is_bug":None,"ai_reason":None,"gpt_judgment":None})
        continue
    if _id.startswith("tr_"):
        gt = gt_map.get(true_id)
        if not gt:
            continue
        if ai_is_bug is not True:
            label = "FN"; gpt_judgment = None
        else:
            prompt = judge_prompt_tpl.format(is_bug=gt["is_bug"], reason=gt["reason"], output=raw.strip())
            resp = client.chat.completions.create(
                model=model_name, temperature=0.7,
                messages=[{"role":"system","content":"You are a helpful AI judge."},
                          {"role":"user","content":prompt}]
            )
            txt = resp.choices[0].message.content.strip()
            label = "TPC" if txt.lower().startswith("yes") else "TPW"
            gpt_judgment = txt
    elif _id.startswith("fl_"):
        label = "FP" if ai_is_bug else "TN"; gpt_judgment = None
    else:
        continue
    rows.append({"id":_id,"label":label,"ai_is_bug":ai_is_bug,"ai_reason":ai_reason,"gpt_judgment":gpt_judgment})

df_r1 = pd.DataFrame(rows)[["id","label","ai_is_bug","ai_reason","gpt_judgment"]]
df_r1.to_csv(R1_CSV, index=False)
print(f"[Round1] Saved judge CSV → {R1_CSV}")
print("[Round1] Summary:", df_r1["label"].value_counts().to_dict())

# —— 初始化标签字典（为 R2/R3 降级更新做准备）——
labels_dict = {r["id"]: r["label"] for r in df_r1.to_dict(orient="records")}

def summarize_labels(labels_dict):
    from collections import Counter
    cnt = Counter(labels_dict.values())
    for k in ["TN","FP","FN","TPC","TPW","parse_error"]:
        cnt.setdefault(k, 0)
    print(f"FP={cnt['FP']}, TN={cnt['TN']}, FN={cnt['FN']}, TPW={cnt['TPW']}, TPC={cnt['TPC']}")
    return cnt


{'43'}


Round1 Judge: 100%|██████████| 102/102 [01:43<00:00,  1.02s/it]

[Round1] Saved judge CSV → /home/yang3j7/NCF0729/test0923_output/sft_qwen3/diff_models_temp0_reason/r1_judge.csv
[Round1] Summary: {'TPC': 30, 'TN': 28, 'FP': 22, 'TPW': 12, 'FN': 8}


In [6]:
# Cell 4. Round1 结果载入为后续输入（并解析 reason）
# =========================
import json, re, pandas as pd

with open(R1_JSON, "r", encoding="utf-8") as f:
    r1_data = json.load(f)

def _normalize_quotes(s: str) -> str:
    return s.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")

def parse_reason_from_generated(text: str):
    raw = str(text); norm = _normalize_quotes(raw)
    # 尝试末尾 JSON
    m = re.search(r'\{.*\}\s*$', norm, flags=re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            bug_val=None
            for k in ("is_bug","is_ncf_bug","BugExistenceJudgment","BugExistanceJudgment"):
                if k in obj: bug_val=str(obj[k]).strip().lower(); break
            is_bug = None if bug_val is None else (bug_val in ("yes","true","1"))
            reason=""
            for k in ("reason","BugReasoningExplanation","explanation"):
                if k in obj and obj[k] is not None:
                    reason=str(obj[k]).strip(); break
            return is_bug, reason
        except Exception:
            pass
    # 回退
    bug_m = re.search(r'"{0,2}(?:is_bug|is_ncf_bug|BugExist[ae]nceJudgment)"{0,2}\s*:\s*["\']?(yes|no|true|false|1|0)["\']?', norm, flags=re.I)
    is_bug = None
    if bug_m:
        is_bug = bug_m.group(1).strip().lower() in ("yes","true","1")
    r_m = re.search(r'(?i)"{0,2}(?:reason|BugReasoningExplanation|explanation)"{0,2}\s*:\s*(?P<q>["\'])(?P<v>(?:\\.|(?!\1).)*?)\1', norm, flags=re.S)
    reason=""
    if r_m:
        val = r_m.group("v")
        try: reason = json.loads(f'"{val}"')
        except: reason = val
    else:
        u = re.search(r'(?i)"{0,2}(?:reason|BugReasoningExplanation|explanation)"{0,2}\s*:\s*([^\r\n}]+)', norm)
        if u: reason = u.group(1).strip().rstrip(',')
    return is_bug, reason

df_all = pd.DataFrame([{"id": x["id"], "generated": x["generated"]} for x in r1_data])
before = len(df_all)
df_all = df_all[~df_all["id"].apply(lambda s: s.split("_", 1)[-1] in skip_ids)].copy()
print(f"[Filter] skipped {before - len(df_all)} samples (from {before} → {len(df_all)})")

df_all["ai_is_bug"], df_all["ai_reason"] = zip(*df_all["generated"].map(parse_reason_from_generated))
print(df_all.head())


[Filter] skipped 2 samples (from 102 → 100)
     id                                          generated  ai_is_bug  \
0  tr_1  {"is_bug": "No", "reason": "Password entry and...      False   
1  tr_2  {"is_bug": "Yes", "reason": "After renaming a ...       True   
2  tr_3  {"is_bug": "Yes", "reason": "After saving a co...       True   
3  tr_4  {"is_bug": "Yes", "reason": "Saving a new tran...       True   
4  tr_5  {"is_bug": "Yes", "reason": "After selecting a...       True   

                                           ai_reason  
0  Password entry and setup flow behaves as expec...  
1  After renaming a category (Gifts -> Giftstest)...  
2  After saving a contact with End date Apr 22, 2...  
3  Saving a new transaction (Income:Salary, $100)...  
4  After selecting a backup (Apr 21, 2025, 14:12)...  


In [7]:
# Cell 5. Rounds 2-4 统一模型加载（Qwen3），一次加载多轮复用
# —— 提供切换“原始base vs finetune后合并”的注释代码
# —— 设定 tokenizer.model_max_length = 16384
# —— 提供 generate_with_limit() 动态限长
# =========================
import torch
from unsloth import FastLanguageModel
# from peft import PeftModel  # ← 若使用 finetune 适配器，解注此行

R234_MAX_SEQ = 16384
R234_MAX_NEW = 2048  # 默认上限，可在调用处覆盖

# === 选项 A：使用原始 base Qwen3（默认）===
R234_MODEL_NAME = "unsloth/Qwen3-14B"
model_r234, tok_r234 = FastLanguageModel.from_pretrained(
    model_name=R234_MODEL_NAME,
    max_seq_length=R234_MAX_SEQ,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)
model_r234.eval()
FastLanguageModel.for_inference(model_r234)
tok_r234.model_max_length = R234_MAX_SEQ  # 关键：设定 tokenizer 的最长长度

# === 选项 B：使用 finetune 后的 Qwen3（四轮一致）
# （如果你想切换到 finetune 版，请解开下方注释，并把 ADAPTER_DIR 指向你的 LoRA）
# ADAPTER_DIR = "/path/to/your/finetuned/adapter"
# base_model_name = "unsloth/Qwen3-14B"
# model_r234, tok_r234 = FastLanguageModel.from_pretrained(
#     model_name=base_model_name,
#     max_seq_length=R234_MAX_SEQ,
#     dtype=torch.bfloat16,
#     load_in_4bit=False,
# )
# model_r234 = PeftModel.from_pretrained(model_r234, ADAPTER_DIR)
# model_r234 = model_r234.merge_and_unload()
# model_r234.eval()
# FastLanguageModel.for_inference(model_r234)
# tok_r234.model_max_length = R234_MAX_SEQ

import torch

@torch.no_grad()
def generate_with_limit(
    prompt_or_messages,
    max_input_tokens: int,         # 动态输入上限（每轮可传不同值）
    max_new_tokens: int = 256,
    temperature: float = 0.0,
    is_chat: bool = True,
):
    # 1) 准备输入文本
    if is_chat:
        text = tok_r234.apply_chat_template(
            prompt_or_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = str(prompt_or_messages)

    # 2) 动态限长：仅裁剪本次输入，不改全局 config
    inputs = tok_r234(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens,
        padding=False,
    )
    inputs = {k: v.to(model_r234.device) for k, v in inputs.items()}

    # 3) 生成
    outputs = model_r234.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature,
        use_cache=True,
        eos_token_id=tok_r234.eos_token_id,
        pad_token_id=tok_r234.eos_token_id,
    )
    return tok_r234.decode(outputs[0], skip_special_tokens=True).strip()


==((====))==  Unsloth 2025.8.5: Fast Qwen3 patching. Transformers: 4.55.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 6/6 [00:05<00:00,  1.11it/s]


In [ ]:
# Cell 6. Prompt 常量（完全不改）
# =========================
NCF_PROMPT_TEMPLATE = """
You are a mobile-app QA classifier.

### Definition
An **NCF bug (Non-Crashing Functional bug)** means:
A **repeatable user action** that, according to product logic or platform rules,  
**should succeed but leaves the app in an incorrect data or state**, without crashing.

Do **NOT** count purely visual, design, or timing issues unless they cause incorrect state.

---

### NCF Bug Categories
- F1: State-transition failure – list/record not inserted, deleted, or updated  
- F2: Persistence failure – change vanishes after refresh/navigation  
- F3: Cross-screen inconsistency – setting changed in screen A not reflected in B  
- F4: Wrong computation/value – wrong unit, number, timestamp, grouping  
- F5: Mis-routed or unavailable control – tap does nothing or wrong thing  
- F6: Localization failure – chosen locale not fully applied  
- F7: Accessibility regression – screen-reader focus blocked, etc.

### NOT NCF (design / UX) examples 
• **Informational banners, snack-bars, or toasts** that persist in the trace  
  - e.g. “Changes are not backed up”, “Outdated Android WebView”  
• **Telemetry or time-based values that drift on their own**  
  - data-usage counters, CPU %, clocks, GPS readouts, live tickers, etc.  
• **Background or automatic refreshes** while user does nothing  
  - list reorder when sync completes, badge counts changing, etc.  
• **Absence of extra confirmation** for standard actions (Paste, Delete, Sort, etc.)  
  unless the spec explicitly requires it.  
• **Place-holder or hint text** that is supposed to disappear after input  
  (“••••••”, “Enter note here…”, “No items yet”).  
• **Hidden-password dots** until the user taps “Show”.  
• **Duplicate view nodes in the dump** caused by RecyclerView / UIAutomator quirks  
  (identical `bounds` or `NAF` attributes).  
• **Values shown in equivalent formats** (00:00 vs 12:00 AM, 1 ft vs 0.3048 m).  
• **Input validation missing / allows out-of-range values 
  - unless spec explicitly defines the valid range AND the invalid input is saved or causes an incorrect state
  
(If the observed issue matches ONLY one of the bullets above and none of the F-categories,  
`is_ncf_bug` **must** be false.)

### Output format
Output **only** a JSON object, no explanation text.

{{
  "is_ncf_bug": true | false,
  "categories": ["F1", "F2", ...],   // empty if false
  "explanation": "<≤ 60 words reason why it's or isn't NCF>"
}}

### Input

reason: {user_input_text}

Your task:
Judge whether the reason describes an NCF bug,
and if yes, list all matching categories.
""".strip()


ANSWER_VERIFY_PROMPT_TEMPLATE = """
You are an evaluator that checks factual alignment.

## Task
Your job is to determine whether a given textual answer **accurately describes** what happens in a provided UI interaction trace, and whether it **correctly identifies** the bug (if any).

You must rely only on the content shown in the trace.

## Judgment Criteria
Mark the answer as **true** if it reasonably matches what the trace shows or implies.

You should consider it **true** when:
1. The behaviors, screens, or outcomes in the answer are **consistent with** the trace (even if not every step is visible);
2. The answer’s description of the bug is **plausible given the trace** and not contradicted by evidence;
3. The answer does **not introduce major hallucinations** (actions, UI elements, or results clearly absent from the trace).

Mark the answer as **false** only when:
- The trace shows **the opposite** of what the answer claims, or  
- The answer adds **clear hallucinations or contradictions** to the observed sequence, or  
- The answer **misinterprets** what happened in the trace (e.g., claims success where the trace shows failure).

If the trace seems incomplete or does not directly confirm the answer but is still compatible with it, you may judge **true**.

## Output Format
Return **only JSON**, no extra text:

{{
  "is_correct": true | false,
  "explanation": "<brief reason for the decision, ≤60 words>"
}}

## Inputs
trace:
{full_trace_text}

answer:
{user_input_text}

## Additional Information
Current date: Sep 23, 2025

## Final Instruction
Compare the answer against the trace.
If the answer correctly and faithfully reflects what the trace shows—including the bug if it exists—output true; otherwise false, following the JSON format above.
""".strip()


In [9]:
# Cell 7. Round2：NCF 判定（固定读取 parsed reason）
# —— 不调用 GPT judge；仅做标签“降级”：
#    FP→TN；TPC/TPW→FN；TN/FN/parse_error 不变
# =========================
import re, json
from tqdm import tqdm

R2_JSON = OUTPUT_DIR / "r2_ncf.json"

def parse_ncf_fields(text: str):
    t = str(text)
    m_bug = re.findall(r'(?<!\w)"is_ncf_bug"\s*:\s*"?\s*(true|false|yes|no|1|0)\s*"?', t, flags=re.I)
    is_ncf = bool(m_bug and m_bug[-1].lower() in ("true","yes","1"))
    return {"is_ncf_bug": is_ncf}

# 仅对 Round1 声称 is_bug=True 的样本参与（根据解析）
df_r2 = df_all[df_all["ai_is_bug"]==True].copy()
print(f"[Round2] Candidates (ai_is_bug=True): {len(df_r2)}")

# 统一构建消息（Round2 固定用解析后的 reason）
def r2_build_messages(reason_text: str):
    return [
        {"role":"system","content":"You are a precise JSON-only responder. Never add extra text."},
        {"role":"user","content": NCF_PROMPT_TEMPLATE.format(user_input_text = reason_text)}
    ]

raw_out, flags = [], []
for reason in tqdm(df_r2["ai_reason"].fillna("").tolist(), desc="Round2 Inference"):
    msgs = r2_build_messages(reason)
    raw = generate_with_limit(msgs, max_input_tokens=2048, max_new_tokens=2048, temperature=0.0, is_chat=True)
    raw_out.append(raw)
    flags.append(parse_ncf_fields(raw)["is_ncf_bug"])

df_r2["ncf_is_bug"] = flags

# 保存原始输出（可选）
df_r2["ncf_raw"] = raw_out
records_r2 = df_r2[["id","ai_reason","ncf_raw","ncf_is_bug"]].to_dict(orient="records")

with open(R2_JSON, "w", encoding="utf-8") as f:
    json.dump(records_r2, f, ensure_ascii=False, indent=2)
print(f"[Round2] Saved (with raw) → {R2_JSON}")

# —— 标签降级 —— 
changed = 0
for _id, is_true in zip(df_r2["id"], df_r2["ncf_is_bug"]):
    if not is_true:
        old = labels_dict.get(_id)
        if old == "FP":
            labels_dict[_id] = "TN"; changed += 1
        elif old in ("TPC","TPW"):
            labels_dict[_id] = "FN"; changed += 1
print(f"[Round2] Labels updated: {changed}")
summarize_labels(labels_dict)


[Round2] Candidates (ai_is_bug=True): 64


Round2 Inference: 100%|██████████| 64/64 [23:49<00:00, 22.33s/it]

[Round2] Saved (with raw) → /home/yang3j7/NCF0729/test0923_output/sft_qwen3/diff_models_temp0_reason/r2_ncf.json
[Round2] Labels updated: 3
FP=21, TN=29, FN=10, TPW=12, TPC=28


Counter({'TN': 29, 'TPC': 28, 'FP': 21, 'TPW': 12, 'FN': 10, 'parse_error': 0})

In [ ]:
# Cell 9. Round4：Reason vs Trace 一致性校验
# —— 不调用 GPT judge；仍按规则降级
# =========================
R3_JSON = OUTPUT_DIR / "r3_verify.json"

def extract_answer_verdict(text: str):
    t = str(text)
    m_flag = re.findall(r'(?<!\w)"is_correct"\s*:\s*"?\s*(true|false|yes|no|1|0)\s*"?', t, flags=re.I)
    is_correct = bool(m_flag and m_flag[-1].lower() in ("true","yes","1"))
    m_exp = re.findall(r'(?<!\w)"explanation"\s*:\s*"(.*?)"', t, flags=re.I|re.S)
    explanation = (m_exp[-1].strip() if m_exp else "")
    return {"is_correct": is_correct, "explanation": explanation}

def get_trace_by_id(sample_id: str) -> str:
    prefix, num = (str(sample_id).split("_",1) + [""])[:2]
    from_bug = (prefix == "tr")
    path = BUG_FILE if from_bug else BUG_FREE_FILE
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    m = {str(it["id"]): it.get("actions_content","") for it in data if isinstance(it, dict)}
    return m.get(num) or m.get(str(int(num)) if num.isdigit() else "", "")

# 仅对当前仍为“正类”的样本（FP/TPC/TPW）进行一致性校验
alive_now = {i for i,l in labels_dict.items() if l in ("FP","TPC","TPW")}
df_r3 = df_all[df_all["id"].isin(alive_now)].copy()
print(f"[Round4] Candidates: {len(df_r3)}")

def r3_build_messages(reason_text: str, trace_text: str):
    return [
        {"role":"system","content":"You are a precise JSON-only responder. Never add extra text."},
        {"role":"user","content": ANSWER_VERIFY_PROMPT_TEMPLATE.format(
            user_input_text = reason_text,
            full_trace_text = trace_text
        )}
    ]

raw4, corr, exps = [], [], []
for _id in tqdm(df_r3["id"].tolist(), desc="Round4 Inference"):
    reason_text = (df_all.set_index("id").loc[_id, "ai_reason"]
                   if R3_INPUT_SOURCE=="reason"
                   else df_all.set_index("id").loc[_id, "generated"])
    trace_text  = get_trace_by_id(_id)
    msgs = r3_build_messages(reason_text or "", trace_text or "")
    raw = generate_with_limit(msgs, max_input_tokens=12000, max_new_tokens=2048, temperature=0.0, is_chat=True)
    raw4.append(raw)
    parsed = extract_answer_verdict(raw)
    corr.append(parsed["is_correct"])
    exps.append(parsed["explanation"])

df_r3["verify_raw"]        = raw4
df_r3["ver_is_correct"]    = corr
df_r3["ver_explanation"]   = exps

records_r3 = df_r3[["id","verify_raw","ver_is_correct","ver_explanation"]].to_dict(orient="records")
with open(R3_JSON, "w", encoding="utf-8") as f:
    json.dump(records_r3, f, ensure_ascii=False, indent=2)
print(f"[Round4] Saved (with raw & explanation) → {R3_JSON}")

# —— 标签降级：对 ver_is_correct=False 的样本降级 —— 
to_false = set(df_r3[df_r3["ver_is_correct"]==False]["id"].tolist())
changed = 0
for _id in to_false:
    old = labels_dict.get(_id)
    if old == "FP":
        labels_dict[_id] = "TN"; changed += 1
    elif old in ("TPC","TPW"):
        labels_dict[_id] = "FN"; changed += 1

print(f"[Round4] Labels updated: {changed}")
summarize_labels(labels_dict)


[Round4] Candidates: 55


Round4 Inference: 100%|██████████| 55/55 [19:05<00:00, 20.82s/it]

[Round4] Saved (with raw & explanation) → /home/yang3j7/NCF0729/test0923_output/sft_qwen3/diff_models_temp0_reason/r4_verify.json
[Round4] Labels updated: 15
FP=15, TN=35, FN=25, TPW=5, TPC=20


Counter({'TN': 35, 'FN': 25, 'TPC': 20, 'FP': 15, 'TPW': 5, 'parse_error': 0})

In [12]:
# Cell 10. 最终导出与汇总
# =========================
import pandas as pd

FINAL_LABELS_CSV = OUTPUT_DIR / "final_labels.csv"
pd.DataFrame([{"id":k,"label":v} for k,v in labels_dict.items()]).to_csv(FINAL_LABELS_CSV, index=False)
print(f"[Final] Saved labels snapshot → {FINAL_LABELS_CSV}")

print("\n[Final] Distribution:")
_ = summarize_labels(labels_dict)


[Final] Saved labels snapshot → /home/yang3j7/NCF0729/test0923_output/sft_qwen3/diff_models_temp0_reason/final_labels.csv

[Final] Distribution:
FP=15, TN=35, FN=25, TPW=5, TPC=20
